# Causal BC on AntMaze Medium, K=10

In [1]:
import random
import torch
import pickle
import os
import numpy as np
import matplotlib.pyplot as plt

from collections import defaultdict

from causal_gym import AntMazePCH
from causal_rl.algo.imitation.imitate import *
from causal_rl.algo.imitation.finetune import *

<frozen importlib._bootstrap>:241: RuntimeWarning: Your system is avx2 capable but pygame was not built with support for it. The performance of some of your blits could be adversely affected. Consider enabling compile time detection with environment variables like PYGAME_DETECT_AVX2=1 if you are compiling without cross compilation.
/home/et2842/miniconda3/envs/causalenv/lib/python3.11/site-packages/pygame/pkgdata.py:25: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import resource_stream, resource_exists


In [2]:
os.environ['CUDA_VISIBLE_DEVICES'] = '0'
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
device

device(type='cuda')

In [3]:
num_steps = 1000
seed = 0
lookback = 10
hidden_dims = {'O'}

random.seed(seed)
torch.manual_seed(seed)

In [4]:
# for training: regular W, O hidden
train_env = AntMazePCH(env_id='antmaze-medium-navigate-singletask-task1-v0', num_steps=num_steps, expert_mode=True, custom_hidden=hidden_dims, seed=seed)

# for eval: corrupted W, O hidden
eval_env = AntMazePCH(env_id='antmaze-medium-navigate-singletask-task1-v0', num_steps=num_steps, expert_mode=False, seed=seed)

## Causal Graph Analysis

In [5]:
# to save time; conceptually the same
small_steps = lookback + 1
small_env = AntMazePCH(env_id='antmaze-medium-navigate-singletask-task1-v0', num_steps=small_steps, seed=seed)
G = parse_graph(small_env.get_graph)
X_small = {f'X{t}' for t in range(small_steps)}
Y = f'Y{small_steps}'

X = {f'X{t}' for t in range(num_steps)}
obs_prefix = train_env.env.observed_unobserved_vars[0]

In [6]:
Z_sets = find_sequential_pi_backdoor(G, X_small, Y, obs_prefix)

base_step = small_steps - 1
base_Z_set = Z_sets[f'X{base_step}']

for i in range(base_step + 1, num_steps):
    updated_base_Z_set = set()
    for v in base_Z_set:
        updated_base_Z_set.add(f'{v[0]}{int(v[1:]) + i - lookback}')

    Z_sets[f'X{i}'] = updated_base_Z_set

Z_sets['X1']

{'A0', 'A1', 'J0', 'J1', 'L0', 'L1', 'P0', 'P1', 'T0', 'T1', 'X0'}

## Expert Trajectories

In [7]:
with open('/home/et2842/causal/expert_traj_antmed.pkl', 'rb') as f:
    records = pickle.load(f)

print(f'loaded {len(records)} trajectories')

loaded 396046 trajectories


In [8]:
dims = {
    'P': 3,
    # 'O': 4,
    'A': 8,
    'L': 3,
    'T': 3,
    'J': 8,
    'W': 2,
    'X': 8,
}

## Training

In [9]:
hidden_size = 256
lr = 3e-4
batch_size = 2048
patience = 15
num_blocks = 4
epochs = 100
dropout = 0.0

In [10]:
cbc_model, cbc_slots, cbc_Z_trim = train_single_policy_long_horizon(
    records,
    Z_sets,
    dims=dims,
    epochs=epochs,
    include_vars=obs_prefix,
    lookback=lookback,
    continuous=True,
    num_actions=train_env.action_space.shape[0],
    hidden_dim=hidden_size,
    num_blocks=num_blocks,
    dropout=dropout,
    lr=lr,
    batch_size=batch_size,
    patience=patience,
    device=device,
    seed=seed,
    action_bounds=(train_env.action_space.low, train_env.action_space.high)
)

cbc_policy = shared_policy_fn_long_horizon(cbc_model, cbc_slots, cbc_Z_trim, continuous=True, device=device)
cbc_policies = make_shared_policy_dict(cbc_policy)

[LongHorizon] Epoch 1: train loss = 0.067455, val loss = 0.027790.


[LongHorizon] Epoch 2: train loss = 0.022610, val loss = 0.019164.


[LongHorizon] Epoch 3: train loss = 0.017098, val loss = 0.015693.


[LongHorizon] Epoch 4: train loss = 0.014379, val loss = 0.013894.


[LongHorizon] Epoch 5: train loss = 0.012728, val loss = 0.012654.


...

[LongHorizon] Epoch 53: train loss = 0.003152, val loss = 0.004914.


[LongHorizon] Epoch 54: train loss = 0.003131, val loss = 0.004846.


[LongHorizon] Epoch 55: train loss = 0.003130, val loss = 0.004759.


[LongHorizon] Epoch 56: train loss = 0.003032, val loss = 0.004786.


[LongHorizon] Epoch 57: train loss = 0.003019, val loss = 0.004722.


## Evaluation

In [ ]:
num_eval_eps = 1000
cbc_returns = collect_imitator_trajectories(
    env=eval_env,
    policies=cbc_policies,
    num_episodes=num_eval_eps,
    max_steps=num_steps,
    hidden_dims=hidden_dims,
    show_progress=True,
    seed=seed + 90210,
)

len(cbc_returns)

In [ ]:
cbc_episode_rewards = defaultdict(float)
for rec in cbc_returns:
    ep = rec['episode']
    cbc_episode_rewards[ep] += float(rec['reward'])

cbc_rewards = [cbc_episode_rewards[e] for e in range(num_eval_eps)]
sum(cbc_rewards) / num_eval_eps

## Save Model

In [ ]:
SAVE_DIR = '/home/et2842/causal/causalrl/models'
os.makedirs(SAVE_DIR, exist_ok=True)
MODEL_PATH = os.path.join(SAVE_DIR, 'cbc_k10_antmed.pt')

checkpoint = {
    "state_dict": cbc_model.state_dict(),
    "slots": cbc_slots,
    "Z_trim": cbc_Z_trim,
    "dims": dims,
    "lookback": lookback,
    "continuous": True,
    "num_actions": train_env.action_space.shape[0],
    "hidden_dim": hidden_size,
    "num_blocks": num_blocks,
    "dropout": dropout,
    "layernorm": True,
    "final_tanh": True,
    "action_bounds_low": eval_env.action_space.low,
    "action_bounds_high": eval_env.action_space.high,
    "input_dim": int(cbc_model.hidden.in_features),
}

torch.save(checkpoint, MODEL_PATH)
print(f'Saved to: {MODEL_PATH}')

In [ ]:
mean_reward = np.mean(cbc_rewards)
std_reward = np.std(cbc_rewards)

print(f"E[Y]          = {mean_reward:.4f}")
print(f"Std[Y]        = {std_reward:.4f}")
print(f"E[Y] ± Std[Y] = {mean_reward:.4f} ± {std_reward:.4f}")

In [ ]:
# success rate: % of episodes solved in under 1000 steps
ep_lengths = defaultdict(int)
for rec in cbc_returns:
    ep_lengths[rec['episode']] += 1

lengths = np.array([ep_lengths[e] for e in range(num_eval_eps)])
successes = lengths < num_steps
success_rate = successes.mean()
se = np.sqrt(success_rate * (1 - success_rate) / num_eval_eps)

print(f"Success rate   = {100 * success_rate:.2f}% ({successes.sum()}/{num_eval_eps} episodes)")
print(f"Std error      = {100 * se:.2f}%")

In [ ]:
# successful episode lengths
success_lengths = lengths[successes]

if len(success_lengths) > 0:
    print(f"Successful episode lengths (n={len(success_lengths)}):")
    print(f"  Mean   = {np.mean(success_lengths):.2f}")
    print(f"  Std    = {np.std(success_lengths):.2f}")
    print(f"  Median = {np.median(success_lengths):.0f}")
    print(f"  Min    = {np.min(success_lengths)}")
    print(f"  Max    = {np.max(success_lengths)}")
    print(f"  25th%  = {np.percentile(success_lengths, 25):.0f}")
    print(f"  75th%  = {np.percentile(success_lengths, 75):.0f}")
else:
    print("No episodes were solved.")